In [10]:
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision

import matplotlib.pyplot as plt
from build_sam import sam_model_registry
import os



In [2]:
# img resolution
img_resolution = 1024

# Select Proper SAM Size you want
sam = sam_model_registry['sam_encoder_b'](checkpoint='../segment-anything/checkpoints/sam_vit_b_image_encoder.pth', custom_img_size=img_resolution)

Loaded encoder from checkpoint


In [3]:
rand_input = torch.rand(1, 3, 256, 256)
rand_output = sam(rand_input)
print(rand_output.shape)

torch.Size([1, 256, 16, 16])


In [4]:
student_model = sam_model_registry['student_encoder']()


In [5]:
student_output = student_model(rand_input)
print(student_output.shape)

torch.Size([1, 256, 16, 16])


In [6]:
# measure the MSE between the two outputs
mse = torch.nn.MSELoss()
loss = mse(rand_output, student_output)
print(loss.item())

1.0247454643249512


In [ ]:
density_estimator1 = nn.Sequential(
    nn.Conv2d(256, 256, 3, stride=1, padding=1),
    nn.GroupNorm(8, 256),
    nn.ReLU(inplace=True)
)

density_estimator2 = nn.Sequential(
    nn.Conv2d(256, 256, 3, stride=1, padding=1),
    nn.GroupNorm(8, 256),
    nn.ReLU(inplace=True)
)

density_estimator3 = nn.Sequential(
    nn.Conv2d(256, 256, 3, stride=1, padding=1),
    nn.GroupNorm(8, 256),
    nn.ReLU(inplace=True)
)

density_estimator4 = nn.Sequential(
    nn.Conv2d(256, 256, 3, stride=1, padding=1),
    nn.GroupNorm(8, 256),
    nn.ReLU(inplace=True),
    nn.Conv2d(256, 1, 1, stride=1)
)

In [ ]:
student_output = F.interpolate(
    density_estimator1(student_output),
    size=student_output.shape[-1] * 2,
    mode="bilinear",
    align_corners=False,
)
student_output = F.interpolate(
    density_estimator2(student_output),
    size=student_output.shape[-1] * 2,
    mode="bilinear",
    align_corners=False,
)
student_output = F.interpolate(
    density_estimator3(student_output),
    size=student_output.shape[-1] * 2,
    mode="bilinear",
    align_corners=False,
)
student_output = F.interpolate(
    density_estimator4(student_output),
    size=student_output.shape[-1] * 2,
    mode="bilinear",
    align_corners=False,
)

In [16]:
print(student_output.shape)

torch.Size([1, 1, 256, 256])


In [19]:
a = torch.rand(1,3,256,1)
b = torch.rand(1,1,256,256)

dist_sq = ((a - b) ** 2)
print(dist_sq.shape)

torch.Size([1, 3, 256, 256])


In [20]:
import torch

def batched_image_points_to_patch_embeddings(
    points: torch.Tensor,
    image_size: tuple = (224, 224),
    patch_size: int = 16
) -> torch.Tensor:
    """
    Map batched input points (h, w) to their Vision Transformer patch embedding indices.
    
    Args:
        points: Tensor of shape (batch_size, num_points, 2) containing (h, w) coordinates.
        image_size: (height, width) of the input image.
        patch_size: Size of each patch (e.g., 16 for ViT-Base). Can also be a tuple (ph, pw).
    
    Returns:
        Tensor of shape (batch_size, num_points, 2) with patch indices (row, col).
    """
    # Ensure valid input shape
    assert points.dim() == 3 and points.shape[-1] == 2, \
        "Input must be of shape (batch_size, num_points, 2)."
    
    # Convert patch_size to (ph, pw)
    if isinstance(patch_size, int):
        ph, pw = patch_size, patch_size
    else:
        ph, pw = patch_size
    
    # Clamp coordinates to valid image bounds [0, H-1] and [0, W-1]
    h, w = image_size
    points_clamped = points.clone()
    points_clamped[..., 0] = torch.clamp(points[..., 0], 0, h - 1)  # Clamp height
    points_clamped[..., 1] = torch.clamp(points[..., 1], 0, w - 1)  # Clamp width
    
    # Compute patch indices using floor division
    patch_indices = torch.div(
        points_clamped,
        torch.tensor([ph, pw], dtype=torch.float32, device=points.device),
        rounding_mode="floor"
    ).long()  # Convert to integer indices
    
    return patch_indices

In [21]:
bs = 2
num_points = 3
points = torch.tensor([
    # Batch 1: Points (h, w)
    [[50, 30], [60, 70], [100, 120]],
    # Batch 2: Points (h, w)
    [[10, 15], [200, 200], [220, 220]]
], dtype=torch.float32)

print(points.shape)

patch_indices = batched_image_points_to_patch_embeddings(points, image_size=(256, 256), patch_size=16)
print(patch_indices.shape)

print(patch_indices)

torch.Size([2, 3, 2])
torch.Size([2, 3, 2])
tensor([[[ 3,  1],
         [ 3,  4],
         [ 6,  7]],

        [[ 0,  0],
         [12, 12],
         [13, 13]]])


In [22]:
student_output = torch.rand(2, 256, 16, 16)
examples = torch.gather(student_output, 2, patch_indices.unsqueeze(1).expand(-1, student_output.size(1), -1, -1))
print(examples.shape)

torch.Size([2, 256, 3, 2])


In [29]:


def extract_patch_features(F, patch_indices):
    """
    Extract features from patch embeddings using the given patch indices.
    
    Args:
        F: Feature tensor of shape (bs, 256, 16, 16).
        patch_indices: Tensor of shape (bs, num_points, 2) containing (row, col) indices.
    
    Returns:
        Extracted features of shape (bs, num_points, 256).
    """
    bs, num_points = patch_indices.shape[:2]
    
    # Unpack row and column indices
    row_indices = patch_indices[..., 0]  # (bs, num_points)
    col_indices = patch_indices[..., 1]  # (bs, num_points)

    # Use advanced indexing to extract features
    features = F[
        torch.arange(bs).unsqueeze(1),  # Batch indices (bs, 1) -> broadcast
        :,  # Keep all 256 channels
        row_indices,  # Row indices (bs, num_points)
        col_indices   # Column indices (bs, num_points)
    ]  # Result: (bs, 256, num_points)

    # Transpose to match desired shape (bs, num_points, 256)
    features = features.permute(0, 2, 1)  # Change (bs, 256, num_points) to (bs, num_points, 256)

    return features

# Example usage
bs = 2
F = torch.randn(bs, 256, 16, 16)  # Feature tensor
points = torch.tensor([[[30, 50], [100, 150], [200, 220]],  # Batch 1 points
                       [[10, 20], [50, 60], [80, 90]]])  # Batch 2 points

patch_indices = batched_image_points_to_patch_embeddings(points, image_size=(256, 256), patch_size=16)
features = extract_patch_features(F, patch_indices)

print(features.shape)  # Expected output: (bs, 3, 256)


torch.Size([2, 256, 3])


In [31]:
import torch

def extract_patch_features(F, patch_indices):
    """
    Extract features from patch embeddings using the given patch indices.
    
    Args:
        F: Feature tensor of shape (bs, 256, 16, 16).
        patch_indices: Tensor of shape (bs, num_points, 2) containing (row, col) indices.
    
    Returns:
        Extracted features of shape (bs, num_points, 256).
    """
    bs, num_points = patch_indices.shape[:2]
    
    # Unpack row and column indices
    row_indices = patch_indices[..., 0]  # (bs, num_points)
    col_indices = patch_indices[..., 1]  # (bs, num_points)

    # Use advanced indexing to extract features
    features = F[
        torch.arange(bs).unsqueeze(1),  # Batch indices (bs, 1) -> broadcast
        :,  # Keep all 256 channels
        row_indices,  # Row indices (bs, num_points)
        col_indices   # Column indices (bs, num_points)
    ]  # Result: (bs, 256, num_points)

    # Transpose to match desired shape (bs, num_points, 256)
    return features.permute(0, 2, 1)

# Example usage
bs = 2
F = torch.randn(bs, 256, 16, 16)  # Feature tensor
points = torch.tensor([[[30, 50], [100, 150], [200, 220]],  # Batch 1 points
                       [[10, 20], [50, 60], [80, 90]]])  # Batch 2 points

patch_indices = batched_image_points_to_patch_embeddings(points, image_size=(224, 224), patch_size=16)
features = extract_patch_features(F, patch_indices)

print(features.shape)  # Expected output: (bs, 3, 256)


torch.Size([2, 256, 3])
